# Classificação de sentimentos com Llama 3.2 1B — SST-2

**Disciplina:** Fundamentos de LLMs — UFPR. **Experimento:** *transfer learning* com backbone congelado e treinamento da cabeça de classificação.

Este notebook reúne a execução experimental realizada no Google Colab com GPU Tesla T4, incluindo a investigação do overflow de gradientes em FP16 e os resultados obtidos após três épocas. Parte do material da disciplina e documenta as adaptações efetuadas.

**Como executar:** abra no Colab com GPU, instale as dependências se necessário, obtenha autorização de acesso ao modelo `meta-llama/Llama-3.2-1B` na Hugging Face e execute em ordem. A autenticação é solicitada por `getpass` (não escreva tokens em células).

**Importante:** os resultados mostrados nas saídas correspondem à execução registrada; novas execuções podem apresentar variações, pois a inicialização aleatória da nova cabeça não foi totalmente controlada antes do carregamento do modelo. O `torch.manual_seed(42)` é aplicado antes do treinamento, mas não antes da inicialização dessa cabeça.

In [1]:

# ============================================================
# DIAGNÓSTICO DO AMBIENTE EXPERIMENTAL
# Verificação dos recursos disponíveis no Google Colab
# ============================================================

import torch

# Identifica a versão do PyTorch.
print("Versão do PyTorch:", torch.__version__)

# Verifica se existe uma GPU CUDA disponível.
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():

    # Identifica o modelo da GPU.
    print("GPU:", torch.cuda.get_device_name(0))

    # Consulta as características de memória da GPU.
    gpu = torch.cuda.get_device_properties(0)

    # Converte bytes para gibibytes (GiB).
    memoria_gib = gpu.total_memory / (1024 ** 3)

    print(f"Memória total da GPU: {memoria_gib:.2f} GiB")

    # Verifica se a GPU possui suporte a bfloat16.
    print("Suporte a bfloat16:", torch.cuda.is_bf16_supported())

else:
    print("Nenhuma GPU CUDA detectada.")


Versão do PyTorch: 2.14.1+cu130
CUDA disponível: True
GPU: Tesla T4
Memória total da GPU: 14.56 GiB
Suporte a bfloat16: True


In [ ]:
# Se o ambiente Colab não tiver as dependências necessárias, descomente:
# %pip install -q -U transformers datasets accelerate

In [2]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")

Hugging Face token: ··········


In [4]:
import torch
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer carregado")
print("Vocabulario:", tokenizer.vocab_size)

Tokenizer carregado
Vocabulario: 128000


### Padding

Para formar um batch, precisamos que as sequências tenham o mesmo tamanho. O Llama não possui originalmente um token de padding separado, então vamos usar o token de fim de texto como padding.

In [5]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

PAD token: <|end_of_text|>
PAD token ID: 128001


In [6]:
text = "This movie is absolutely fantastic!"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer(text)["input_ids"]

print("Texto:")
print(text)
print("\nTokens:")
print(tokens)
print("\nIDs:")
print(token_ids)

Texto:
This movie is absolutely fantastic!

Tokens:
['This', 'Ġmovie', 'Ġis', 'Ġabsolutely', 'Ġfantastic', '!']

IDs:
[128000, 2028, 5818, 374, 11112, 14964, 0]


In [9]:
encoded = tokenizer(text)
decoded = tokenizer.decode(
    encoded["input_ids"],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False)

print("Original:", text)
print("Decodificado:", decoded)

Original: This movie is absolutely fantastic!
Decodificado: This movie is absolutely fantastic!


In [8]:

frases = [
    "This movie is great!",
    "This movie was surprisingly good and very entertaining!"
]

resultado = tokenizer(
    frases,
    padding=True,
    return_tensors="pt"
)

print("Input IDs:")
print(resultado["input_ids"])

print("\nAttention mask:")
print(resultado["attention_mask"])

print("\nDimensões:", resultado["input_ids"].shape)


Input IDs:
tensor([[128000,   2028,   5818,    374,   2294,      0, 128001, 128001, 128001,
         128001],
        [128000,   2028,   5818,    574,  29392,   1695,    323,   1633,  30311,
              0]])

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Dimensões: torch.Size([2, 10])


## Modelo de classificação: substituindo a cabeça de linguagem

O Llama pré-treinado é originalmente um modelo causal, utilizado para prever o próximo token. A classe `AutoModelForSequenceClassification` reutiliza o Transformer e acrescenta uma nova cabeça de classificação binária: `0 = negative` e `1 = positive`.

**Neste experimento, treinaremos somente a nova cabeça (`model.score`) e manteremos o backbone congelado.** Isto é diferente de um fine-tuning completo, que será objeto de outra investigação.

In [10]:
print("Capacidade CUDA:", torch.cuda.get_device_capability(0))
print(
    "BF16 nativo:",
    torch.cuda.is_bf16_supported(including_emulation=False)
)

Capacidade CUDA: (7, 5)
BF16 nativo: False


In [11]:

from transformers import AutoModelForSequenceClassification

# Carrega o Transformer pré-treinado e prepara
# uma nova cabeça de classificação binária.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    dtype=torch.float16,  # Adaptado para a GPU Tesla T4
    device_map="auto")

# Informa ao modelo qual ID representa padding.
model.config.pad_token_id = tokenizer.pad_token_id

print("Modelo carregado")
print("Dispositivo:", model.device)
print("Tipo numérico:", next(model.parameters()).dtype)


Modelo carregado
Dispositivo: cuda:0
Tipo numérico: torch.float16


## Conhecendo o modelo

In [12]:
print("Hidden size:", model.config.hidden_size)
print("Número de camadas:", model.config.num_hidden_layers)
print("Número de labels:", model.config.num_labels)
print("Tipo do modelo:", type(model).__name__)

Hidden size: 2048
Número de camadas: 16
Número de labels: 2
Tipo do modelo: LlamaForSequenceClassification


### Quantos parâmetros serão treinados?

In [13]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Parâmetros totais:      {total_params:,}")
print(f"Parâmetros treináveis:  {trainable_params:,}")

Parâmetros totais:      1,235,818,496
Parâmetros treináveis:  1,235,818,496


In [14]:
print("Cabeça de classificação:", model.score)
print("Dimensões dos pesos:", model.score.weight.shape)
print("Parâmetros da cabeça:", model.score.weight.numel())

Cabeça de classificação: Linear(in_features=2048, out_features=2, bias=False)
Dimensões dos pesos: torch.Size([2, 2048])
Parâmetros da cabeça: 4096


In [15]:

from datasets import load_dataset
from collections import Counter

# Carrega o conjunto de dados SST-2.
dataset = load_dataset("stanfordnlp/sst2")

# Embaralha os dados com uma semente fixa e
# seleciona 1.000 exemplos de treinamento.
train = dataset["train"].shuffle(seed=42).select(range(1000))

# Mantém a divisão de validação original.
validation = dataset["validation"]

print("Treinamento:", len(train))
print("Validação:", len(validation))

# Verifica a distribuição das duas classes.
print("Classes de treinamento:", Counter(train["label"]))
print("Classes de validação:", Counter(validation["label"]))


Treinamento: 1000
Validação: 872
Classes de treinamento: Counter({1: 557, 0: 443})
Classes de validação: Counter({1: 444, 0: 428})


## Observando os dados de treinamento

In [16]:
label_names = {
    0: "negative",
    1: "positive"
}

for i in range(10):
    print(f"Exemplo {i + 1}")
    print("Texto:", train[i]["sentence"])
    print("Label:", train[i]["label"], "(", label_names[train[i]["label"]], ")")
    print()

Exemplo 1
Texto: klein , charming in comedies like american pie and dead-on in election , 
Label: 1 ( positive )

Exemplo 2
Texto: be fruitful 
Label: 1 ( positive )

Exemplo 3
Texto: soulful and 
Label: 1 ( positive )

Exemplo 4
Texto: the proud warrior that still lingers in the souls of these characters 
Label: 1 ( positive )

Exemplo 5
Texto: covered earlier and much better 
Label: 0 ( negative )

Exemplo 6
Texto: wise and powerful 
Label: 1 ( positive )

Exemplo 7
Texto: a powerful and reasonably fulfilling gestalt 
Label: 1 ( positive )

Exemplo 8
Texto: smart and newfangled 
Label: 1 ( positive )

Exemplo 9
Texto: it too is a bomb . 
Label: 0 ( negative )

Exemplo 10
Texto: guilty about it 
Label: 0 ( negative )



## Tokenização do SST-2

Cada frase é convertida em `input_ids` e `attention_mask`, com comprimento máximo de 128 tokens. Usamos padding até o máximo para manter a configuração do experimento realizado em aula; o custo de padding poderá ser investigado depois.

In [17]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train = train.map(tokenize, batched=True)
validation = validation.map(tokenize, batched=True)

In [18]:

# Inspeciona o primeiro exemplo tokenizado.
exemplo = train[0]

# Quantidade de posições na sequência.
comprimento = len(exemplo["input_ids"])

# Quantidade de posições efetivamente utilizadas.
tokens_validos = sum(exemplo["attention_mask"])

# Posições adicionadas para completar o tamanho máximo.
tokens_padding = comprimento - tokens_validos

print("Texto original:", exemplo["sentence"])
print("Classe:", exemplo["label"])
print("Comprimento:", comprimento)
print("Tokens válidos:", tokens_validos)
print("Posições de padding:", tokens_padding)

print("\nPrimeiros 15 IDs:")
print(exemplo["input_ids"][:15])

print("\nPrimeiros 15 valores da máscara:")
print(exemplo["attention_mask"][:15])


Texto original: klein , charming in comedies like american pie and dead-on in election , 
Classe: 1
Comprimento: 128
Tokens válidos: 18
Posições de padding: 110

Primeiros 15 IDs:
[128000, 23960, 258, 1174, 35509, 304, 63113, 552, 1093, 39542, 4447, 323, 5710, 10539, 304]

Primeiros 15 valores da máscara:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [19]:
print(train.column_names)
print("\nPrimeiro exemplo tokenizado:")
print(train[0])

['idx', 'sentence', 'label', 'input_ids', 'attention_mask']

Primeiro exemplo tokenizado:
{'idx': 32326, 'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1, 'input_ids': [128000, 23960, 258, 1174, 35509, 304, 63113, 552, 1093, 39542, 4447, 323, 5710, 10539, 304, 6355, 1174, 220, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,

In [20]:
train.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

validation.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [21]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [22]:

# Quantidade de lotes em cada conjunto.
print("Lotes de treinamento:", len(train_loader))
print("Lotes de validação:", len(validation_loader))

# Obtém um único lote, sem realizar treinamento.
batch = next(iter(train_loader))

print("\nCampos disponíveis:", batch.keys())

print("Input IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Labels:", batch["label"].shape)

print("\nTipos de dados:")
for nome, tensor in batch.items():
    print(nome, tensor.dtype)


Lotes de treinamento: 125
Lotes de validação: 109

Campos disponíveis: dict_keys(['label', 'input_ids', 'attention_mask'])
Input IDs: torch.Size([8, 128])
Attention mask: torch.Size([8, 128])
Labels: torch.Size([8])

Tipos de dados:
label torch.int64
input_ids torch.int64
attention_mask torch.int64


In [23]:

# ============================================================
# TESTE DE FORWARD PASS — SEM TREINAMENTO
# ============================================================

# Coloca o modelo em modo de avaliação.
model.eval()

# Reinicia a medição do pico de memória.
torch.cuda.reset_peak_memory_stats()

# Seleciona duas avaliações.
batch = next(iter(validation_loader))

input_ids = batch["input_ids"][:2].to(model.device)
attention_mask = batch["attention_mask"][:2].to(model.device)
labels = batch["label"][:2].to(model.device)

# Executa o modelo sem registrar gradientes.
with torch.inference_mode():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

torch.cuda.synchronize()

# Analisa as saídas.
print("Formato dos logits:", outputs.logits.shape)
print("Logits:", outputs.logits)
print("Labels reais:", labels)
print("Loss antes do treinamento:", outputs.loss.item())

# Consulta a memória utilizada.
memoria = torch.cuda.memory_allocated() / 1024**3
pico = torch.cuda.max_memory_allocated() / 1024**3
livre, total = torch.cuda.mem_get_info()

print(f"\nMemória alocada: {memoria:.2f} GiB")
print(f"Pico de memória: {pico:.2f} GiB")
print(f"Memória livre na GPU: {livre / 1024**3:.2f} GiB")


Formato dos logits: torch.Size([2, 2])
Logits: tensor([[1.0234, 2.5820],
        [1.2471, 2.0000]], device='cuda:0', dtype=torch.float16)
Labels reais: tensor([1, 0], device='cuda:0')
Loss antes do treinamento: 0.6650390625

Memória alocada: 2.32 GiB
Pico de memória: 2.33 GiB
Memória livre na GPU: 12.08 GiB


## Configuração do experimento: treinar apenas a cabeça

O backbone possui mais de 1,2 bilhão de parâmetros. Congelamos o Transformer e atualizamos apenas os 4.096 pesos de `model.score`. A cabeça usa FP32 e o restante do modelo é carregado em FP16 para caber na Tesla T4.

In [24]:

# Guarda os pesos iniciais da cabeça para permitir
# comparações posteriores com a mesma inicialização.
pesos_iniciais_cabeca = {
    nome: tensor.detach().cpu().clone()
    for nome, tensor in model.score.state_dict().items()
}

# Congela todos os parâmetros do modelo.
for parametro in model.parameters():
    parametro.requires_grad = False

# Reabilita os gradientes apenas na cabeça.
for parametro in model.score.parameters():
    parametro.requires_grad = True

# Confere quantos parâmetros ainda poderão ser treinados.
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Parâmetros totais: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")
print(f"Percentual treinável: {100 * trainable_params / total_params:.6f}%")


Parâmetros totais: 1,235,818,496
Parâmetros treináveis: 4,096
Percentual treinável: 0.000331%


In [25]:

# ============================================================
# EXPERIMENTO B — TRANSFER LEARNING
# Preparação do otimizador com backbone congelado
# ============================================================

# Mantém os pesos da cabeça em precisão FP32.
model.score.float()

# AdamW atualizará SOMENTE os parâmetros da cabeça.
optimizer = torch.optim.AdamW(
    model.score.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

# Controla a escala dos gradientes em operações FP16.
scaler = torch.amp.GradScaler("cuda")

# Verificações de segurança.
print("Precisão do backbone:",
      next(model.model.parameters()).dtype)

print("Precisão da cabeça:",
      model.score.weight.dtype)

print("Parâmetros no otimizador:",
      sum(
          p.numel()
          for grupo in optimizer.param_groups
          for p in grupo["params"]
      ))

print("GradScaler habilitado:", scaler.is_enabled())


Precisão do backbone: torch.float16
Precisão da cabeça: torch.float32
Parâmetros no otimizador: 4096
GradScaler habilitado: True


## Diagnóstico numérico — primeira tentativa

A primeira tentativa utiliza o `GradScaler` com escala inicial padrão (65.536). Na execução documentada, a loss foi finita, mas os gradientes resultaram em `inf` e o passo foi ignorado. Essa célula preserva **a evidência do diagnóstico**, não deve ser confundida com uma época de treinamento.

In [26]:

# ============================================================
# TESTE CONTROLADO — UM PASSO DE TREINAMENTO
# Backbone congelado + cabeça FP32
# ============================================================

import torch.nn.functional as F

model.train()

# Guarda uma cópia dos pesos para restaurá-los depois.
pesos_antes = model.score.weight.detach().clone()

# Seleciona apenas dois exemplos do primeiro lote.
batch = next(iter(train_loader))

input_ids = batch["input_ids"][:2].to(model.device)
attention_mask = batch["attention_mask"][:2].to(model.device)
labels = batch["label"][:2].to(model.device)

# Zera gradientes e inicia a medição de memória.
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()

# 1. Passagem para frente (forward).
with torch.autocast("cuda", dtype=torch.float16):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

# 2. Calcula a perda em FP32.
loss = F.cross_entropy(outputs.logits.float(), labels)

print(f"Loss inicial: {loss.item():.6f}")

if not torch.isfinite(loss).item():
    raise RuntimeError("Loss não finita. Treinamento interrompido.")

# 3. Retropropagação (backward).
scaler.scale(loss).backward()

# Desfaz a escala dos gradientes para inspecioná-los.
scaler.unscale_(optimizer)

gradientes = model.score.weight.grad

print(
    "Gradientes finitos:",
    torch.isfinite(gradientes).all().item()
)
print(
    f"Norma dos gradientes: {gradientes.norm().item():.6f}"
)

# 4. Atualiza os pesos.
scaler.step(optimizer)
scaler.update()

# 5. Verifica se os pesos efetivamente mudaram.
alteracao = (
    model.score.weight.detach() - pesos_antes
).abs().max().item()

print(f"Maior alteração de peso: {alteracao:.8f}")

# 6. Consulta o pico de memória.
pico = torch.cuda.max_memory_allocated() / 1024**3

print(f"Pico de memória: {pico:.2f} GiB")

# 7. Restaura os pesos anteriores ao teste.
with torch.no_grad():
    model.score.weight.copy_(pesos_antes)

# Reinicializa os estados do otimizador e do GradScaler.
optimizer = torch.optim.AdamW(
    model.score.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")

print("Pesos restaurados:",
      torch.equal(model.score.weight, pesos_antes))


Loss inicial: 1.670754
Gradientes finitos: False
Norma dos gradientes: inf
Maior alteração de peso: 0.00000000
Pico de memória: 2.34 GiB
Pesos restaurados: True


### Hipótese testada: reduzir a escala inicial

Repetimos o mesmo lote de dois exemplos com `init_scale=128`. Os gradientes ficaram finitos e os pesos foram atualizados. Em ambos os testes os pesos foram restaurados para que o treinamento efetivo começasse do mesmo estado.

In [27]:

# ============================================================
# DIAGNÓSTICO — GradScaler com escala inicial reduzida
# ============================================================

import torch.nn.functional as F

model.train()

# Guarda os pesos atuais para restauração.
pesos_antes = model.score.weight.detach().clone()

# Reutiliza input_ids, attention_mask e labels
# do teste anterior, preservando os mesmos exemplos.

# Reduz a escala inicial de 65536 para 128.
scaler_teste = torch.amp.GradScaler(
    "cuda",
    init_scale=128.0
)

optimizer.zero_grad(set_to_none=True)

# Forward em precisão mista.
with torch.autocast("cuda", dtype=torch.float16):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

loss = F.cross_entropy(outputs.logits.float(), labels)

print(f"Loss: {loss.item():.6f}")
print("Escala inicial:", scaler_teste.get_scale())

# Calcula os gradientes.
scaler_teste.scale(loss).backward()
scaler_teste.unscale_(optimizer)

grad = model.score.weight.grad
grad_ok = torch.isfinite(grad).all().item()

print("Gradientes finitos:", grad_ok)

if grad_ok:
    print(f"Norma dos gradientes: {grad.norm().item():.6f}")

# O GradScaler pula o passo caso detecte inf/NaN.
scaler_teste.step(optimizer)
scaler_teste.update()

print("Escala após o passo:", scaler_teste.get_scale())

alteracao = (
    model.score.weight.detach() - pesos_antes
).abs().max().item()

print(f"Maior alteração de peso: {alteracao:.8f}")

# Restaura os pesos e reinicializa o otimizador.
with torch.no_grad():
    model.score.weight.copy_(pesos_antes)

optimizer = torch.optim.AdamW(
    model.score.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")

print("Pesos restaurados:",
      torch.equal(model.score.weight, pesos_antes))


Loss: 1.670754
Escala inicial: 128.0
Gradientes finitos: True
Norma dos gradientes: 99.741760
Escala após o passo: 128.0
Maior alteração de peso: 0.00002001
Pesos restaurados: True


## Treinamento e validação — três épocas

Experimento executado com 1.000 exemplos de treinamento, 872 de validação, batch de 8, AdamW (`lr=2e-5`, `weight_decay=0.01`), autocast FP16 e `GradScaler(init_scale=128)`. O backbone permanece congelado. Após cada época calculamos a acurácia, loss e matriz de confusão no conjunto de validação.

In [28]:

# ============================================================
# EXPERIMENTO B — PRIMEIRA ÉPOCA DE TRANSFER LEARNING
# Backbone congelado; somente a cabeça será treinada
# ============================================================

from tqdm.auto import tqdm
import torch.nn.functional as F

# Reprodutibilidade da sequência de lotes.
torch.manual_seed(42)

# Confirma que apenas a cabeça será treinada.
assert sum(
    p.numel() for p in model.parameters() if p.requires_grad
) == 4096

# Reinicia o otimizador e o escalonador.
optimizer = torch.optim.AdamW(
    model.score.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=128.0
)

model.train()
torch.cuda.reset_peak_memory_stats()

soma_loss = 0.0
acertos = 0
total = 0
passos_pulados = 0

for batch in tqdm(train_loader, desc="Época 1"):

    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    labels = batch["label"].to(model.device)

    optimizer.zero_grad(set_to_none=True)

    # Forward com autocast FP16.
    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    # Cross-entropy calculada em FP32.
    loss = F.cross_entropy(outputs.logits.float(), labels)

    if not torch.isfinite(loss).item():
        raise RuntimeError("Loss não finita durante o treinamento.")

    # Backward e atualização.
    scaler.scale(loss).backward()

    escala_anterior = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < escala_anterior:
        passos_pulados += 1

    # Acumula métricas de treinamento.
    quantidade = labels.size(0)
    soma_loss += loss.item() * quantidade
    acertos += (
        outputs.logits.argmax(dim=1) == labels
    ).sum().item()
    total += quantidade

print("\nRESULTADOS — ÉPOCA 1")
print(f"Exemplos processados: {total}")
print(f"Loss média: {soma_loss / total:.6f}")
print(f"Acurácia de treinamento: {100 * acertos / total:.2f}%")
print(f"Passos pulados por overflow: {passos_pulados}")
print(f"Escala final: {scaler.get_scale()}")
print(
    "Pico de memória:",
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB"
)



RESULTADOS — ÉPOCA 1
Exemplos processados: 1000
Loss média: 0.863192
Acurácia de treinamento: 44.00%
Passos pulados por overflow: 0
Escala final: 128.0
Pico de memória: 2.44 GiB


In [29]:

# ============================================================
# EXPERIMENTO B — VALIDAÇÃO APÓS A PRIMEIRA ÉPOCA
# ============================================================

import torch.nn.functional as F
from tqdm.auto import tqdm

# Modo de avaliação: desativa comportamentos de treinamento.
model.eval()

total_loss = 0.0
acertos = 0
total = 0

# Linhas = classe real; colunas = classe prevista.
matriz_confusao = torch.zeros((2, 2), dtype=torch.long)

# Não precisamos calcular gradientes na validação.
with torch.inference_mode():

    for batch in tqdm(validation_loader, desc="Validação"):

        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # Calcula a perda em FP32, somando por exemplo.
        logits = outputs.logits.float()
        loss = F.cross_entropy(
            logits, labels, reduction="sum"
        )

        previsoes = logits.argmax(dim=1)

        total_loss += loss.item()
        acertos += (previsoes == labels).sum().item()
        total += labels.size(0)

        # Contabiliza as quatro combinações:
        # real 0 / previsto 0, real 0 / previsto 1,
        # real 1 / previsto 0, real 1 / previsto 1.
        indices = labels * 2 + previsoes

        matriz_confusao += torch.bincount(
            indices.cpu(),
            minlength=4
        ).reshape(2, 2)

print("\nRESULTADOS DE VALIDAÇÃO")
print("Exemplos avaliados:", total)
print(f"Loss média: {total_loss / total:.6f}")
print(f"Acurácia: {100 * acertos / total:.2f}%")

print("\nMatriz de confusão:")
print(matriz_confusao)

print("\nDistribuição das previsões:")
print("Negativas:", matriz_confusao[:, 0].sum().item())
print("Positivas:", matriz_confusao[:, 1].sum().item())

print("\nBaseline de validação: 50.92%")



RESULTADOS DE VALIDAÇÃO
Exemplos avaliados: 872
Loss média: 0.766756
Acurácia: 50.80%

Matriz de confusão:
tensor([[ 74, 354],
        [ 75, 369]])

Distribuição das previsões:
Negativas: 149
Positivas: 723

Baseline de validação: 50.92%


In [30]:

# ============================================================
# EXPERIMENTO B — SEGUNDA ÉPOCA DE TREINAMENTO
# Continuação do treinamento da cabeça
# ============================================================

model.train()

# Confere o estado preservado do experimento.
print("Taxa de aprendizado:", optimizer.param_groups[0]["lr"])
print("Escala inicial:", scaler.get_scale())

soma_loss = 0.0
acertos = 0
total = 0
passos_pulados = 0

for batch in tqdm(train_loader, desc="Época 2"):

    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    labels = batch["label"].to(model.device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    loss = F.cross_entropy(
        outputs.logits.float(),
        labels
    )

    if not torch.isfinite(loss).item():
        raise RuntimeError("Loss não finita.")

    scaler.scale(loss).backward()

    escala_anterior = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < escala_anterior:
        passos_pulados += 1

    quantidade = labels.size(0)

    soma_loss += loss.item() * quantidade
    acertos += (
        outputs.logits.argmax(dim=1) == labels
    ).sum().item()

    total += quantidade

print("\nRESULTADOS — ÉPOCA 2")
print(f"Loss média: {soma_loss / total:.6f}")
print(f"Acurácia de treinamento: {100 * acertos / total:.2f}%")
print(f"Passos pulados: {passos_pulados}")
print(f"Escala final: {scaler.get_scale()}")


Taxa de aprendizado: 2e-05
Escala inicial: 128.0



RESULTADOS — ÉPOCA 2
Loss média: 0.805897
Acurácia de treinamento: 46.70%
Passos pulados: 0
Escala final: 128.0


In [31]:

# ============================================================
# EXPERIMENTO B — VALIDAÇÃO APÓS A SEGUNDA ÉPOCA
# ============================================================

import torch.nn.functional as F
from tqdm.auto import tqdm

# Modo de avaliação: desativa comportamentos de treinamento.
model.eval()

total_loss = 0.0
acertos = 0
total = 0

# Linhas = classe real; colunas = classe prevista.
matriz_confusao = torch.zeros((2, 2), dtype=torch.long)

# Não precisamos calcular gradientes na validação.
with torch.inference_mode():

    for batch in tqdm(validation_loader, desc="Validação"):

        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # Calcula a perda em FP32, somando por exemplo.
        logits = outputs.logits.float()
        loss = F.cross_entropy(
            logits, labels, reduction="sum"
        )

        previsoes = logits.argmax(dim=1)

        total_loss += loss.item()
        acertos += (previsoes == labels).sum().item()
        total += labels.size(0)

        # Contabiliza as quatro combinações:
        # real 0 / previsto 0, real 0 / previsto 1,
        # real 1 / previsto 0, real 1 / previsto 1.
        indices = labels * 2 + previsoes

        matriz_confusao += torch.bincount(
            indices.cpu(),
            minlength=4
        ).reshape(2, 2)

print("\nRESULTADOS DE VALIDAÇÃO")
print("Exemplos avaliados:", total)
print(f"Loss média: {total_loss / total:.6f}")
print(f"Acurácia: {100 * acertos / total:.2f}%")

print("\nMatriz de confusão:")
print(matriz_confusao)

print("\nDistribuição das previsões:")
print("Negativas:", matriz_confusao[:, 0].sum().item())
print("Positivas:", matriz_confusao[:, 1].sum().item())

print("\nBaseline de validação: 50.92%")


RESULTADOS DE VALIDAÇÃO
Exemplos avaliados: 872
Loss média: 0.717696
Acurácia: 53.10%

Matriz de confusão:
tensor([[ 92, 336],
        [ 73, 371]])

Distribuição das previsões:
Negativas: 165
Positivas: 707

Baseline de validação: 50.92%


In [33]:

# ============================================================
# EXPERIMENTO B — TERCEIRA ÉPOCA DE TREINAMENTO
# Continuação do treinamento da cabeça
# ============================================================

model.train()

# Confere o estado preservado do experimento.
print("Taxa de aprendizado:", optimizer.param_groups[0]["lr"])
print("Escala inicial:", scaler.get_scale())

soma_loss = 0.0
acertos = 0
total = 0
passos_pulados = 0

for batch in tqdm(train_loader, desc="Época 3"):

    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    labels = batch["label"].to(model.device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    loss = F.cross_entropy(
        outputs.logits.float(),
        labels
    )

    if not torch.isfinite(loss).item():
        raise RuntimeError("Loss não finita.")

    scaler.scale(loss).backward()

    escala_anterior = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < escala_anterior:
        passos_pulados += 1

    quantidade = labels.size(0)

    soma_loss += loss.item() * quantidade
    acertos += (
        outputs.logits.argmax(dim=1) == labels
    ).sum().item()

    total += quantidade

print("\nRESULTADOS — ÉPOCA 3")
print(f"Loss média: {soma_loss / total:.6f}")
print(f"Acurácia de treinamento: {100 * acertos / total:.2f}%")
print(f"Passos pulados: {passos_pulados}")
print(f"Escala final: {scaler.get_scale()}")


Taxa de aprendizado: 2e-05
Escala inicial: 128.0



RESULTADOS — ÉPOCA 3
Loss média: 0.756613
Acurácia de treinamento: 52.70%
Passos pulados: 0
Escala final: 128.0


In [34]:
# ============================================================
# EXPERIMENTO B — VALIDAÇÃO APÓS A TERCEIRA ÉPOCA
# ============================================================

import torch.nn.functional as F
from tqdm.auto import tqdm

# Modo de avaliação: desativa comportamentos de treinamento.
model.eval()

total_loss = 0.0
acertos = 0
total = 0

# Linhas = classe real; colunas = classe prevista.
matriz_confusao = torch.zeros((2, 2), dtype=torch.long)

# Não precisamos calcular gradientes na validação.
with torch.inference_mode():

    for batch in tqdm(validation_loader, desc="Validação"):

        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # Calcula a perda em FP32, somando por exemplo.
        logits = outputs.logits.float()
        loss = F.cross_entropy(
            logits, labels, reduction="sum"
        )

        previsoes = logits.argmax(dim=1)

        total_loss += loss.item()
        acertos += (previsoes == labels).sum().item()
        total += labels.size(0)

        # Contabiliza as quatro combinações:
        # real 0 / previsto 0, real 0 / previsto 1,
        # real 1 / previsto 0, real 1 / previsto 1.
        indices = labels * 2 + previsoes

        matriz_confusao += torch.bincount(
            indices.cpu(),
            minlength=4
        ).reshape(2, 2)

print("\nRESULTADOS DE VALIDAÇÃO")
print("Exemplos avaliados:", total)
print(f"Loss média: {total_loss / total:.6f}")
print(f"Acurácia: {100 * acertos / total:.2f}%")

print("\nMatriz de confusão:")
print(matriz_confusao)

print("\nDistribuição das previsões:")
print("Negativas:", matriz_confusao[:, 0].sum().item())
print("Positivas:", matriz_confusao[:, 1].sum().item())

print("\nBaseline de validação: 50.92%")


RESULTADOS DE VALIDAÇÃO
Exemplos avaliados: 872
Loss média: 0.657471
Acurácia: 61.12%

Matriz de confusão:
tensor([[295, 133],
        [206, 238]])

Distribuição das previsões:
Negativas: 501
Positivas: 371

Baseline de validação: 50.92%


## Salvamento do checkpoint

O checkpoint salva **apenas os pesos da cabeça**, os estados do AdamW e do `GradScaler`, além de metadados. Os pesos do backbone do Llama não são distribuídos. A execução pressupõe que a validação da terceira época tenha sido realizada na célula imediatamente anterior.

In [35]:

# ============================================================
# CHECKPOINT — TRANSFER LEARNING APÓS 3 ÉPOCAS
# ============================================================

# Execute a célula de validação da terceira época antes deste checkpoint.
arquivo = "checkpoint_transfer_lr2e-5_epoca3.pt"

checkpoint = {
    "epoca": 3,
    "modelo_base": MODEL_ID,

    # Pesos treinados da cabeça de classificação.
    "cabeca": {
        nome: tensor.detach().cpu().clone()
        for nome, tensor in model.score.state_dict().items()
    },

    # Estados necessários para continuar o treinamento.
    "optimizer": optimizer.state_dict(),
    "scaler": scaler.state_dict(),

    "configuracao": {
        "learning_rate": 2e-5,
        "batch_size": 8,
        "amostras_treinamento": 1000,
        "epocas": 3,
        "backbone_congelado": True
    },

    "validacao": {
        "loss": total_loss / total,
        "acuracia": acertos / total,
        "matriz_confusao": matriz_confusao.tolist()
    }
}

torch.save(checkpoint, arquivo)

print("Checkpoint salvo:", arquivo)


Checkpoint salvo: checkpoint_transfer_lr2e-5_epoca3.pt


In [36]:
from google.colab import files
files.download(arquivo)

## Resultados observados e discussão

| Época | Loss de treinamento | Acurácia de treinamento | Loss de validação | Acurácia de validação |
|---:|---:|---:|---:|---:|
| 1 | 0,863192 | 44,00% | 0,766756 | 50,80% |
| 2 | 0,805897 | 46,70% | 0,717696 | 53,10% |
| 3 | 0,756613 | 52,70% | 0,657471 | **61,12%** |

A baseline da validação, prevendo sempre a classe positiva majoritária, é **50,92%**. A terceira época supera essa referência em aproximadamente **10,2 pontos percentuais**, embora o desempenho ainda seja modesto.

Matriz de confusão após a terceira época (linhas = classes reais; colunas = classes previstas):

```text
                 Previsto 0   Previsto 1
Real 0                295          133
Real 1                206          238
```

O recall negativo passou de **17,29%** na época 1 para **68,93%** na época 3; o recall positivo caiu de **83,11%** para **53,60%**. Assim, houve melhora global, mas persistem diferenças relevantes entre classes.

**Limitações metodológicas:** o resultado provém de uma única execução, com uma cabeça inicialmente aleatória, sem controle completo da semente anterior ao carregamento do modelo. A validação foi consultada a cada época e não deve ser tratada como um teste independente final. O split de teste público do SST-2 não oferece rótulos para avaliação direta nesse fluxo. A comparação com outras abordagens deve utilizar condições controladas e, idealmente, uma avaliação independente.

**Próximos estudos:** comparação de taxas de aprendizado, classificação por *prompting* e avaliação da viabilidade de fine-tuning integral na infraestrutura disponível. Essas etapas **ainda não foram realizadas** neste notebook.